### GWAS Validation

In [1]:
import pandas as pd
import requests
import os

data_processed = os.path.expanduser("~/Pan-Autoimmune-miRNA-ML/data/processed")

## Use GWAS Catalog API v2 — more reliable endpoint

def query_gwas_genes(trait):
    url = f"https://www.ebi.ac.uk/gwas/rest/api/efoTraits/search/findByEfoUri?uri={trait}"
    
    # Use the studies endpoint instead
    url = f"https://www.ebi.ac.uk/gwas/rest/api/studies/search/findByDiseaseTrait?diseaseTrait={trait}"
    response = requests.get(url, headers={"Accept": "application/json"}, timeout=15)
    print(f"Status: {response.status_code}")
    if response.status_code == 200:
        data = response.json()
        print(str(data)[:300])
        return data
    return None

result = query_gwas_genes("vitiligo")

Status: 200
{'_embedded': {'studies': [{'initialSampleSize': '1,339 European ancestry cases', 'gxe': False, 'gxg': False, 'snpCount': 520460, 'qualifier': None, 'imputed': False, 'pooled': False, 'studyDesignComment': None, 'accessionId': 'GCST000981', 'fullPvalueSet': False, 'userRequested': False, 'platforms'


In [2]:
## Load our SHAP genes and DEGs for validation
shap_genes = {}
for disease in ["RA", "SLE", "Vitiligo", "T1D"]:
    df = pd.read_csv(f"{data_processed}/{disease}_top_SHAP_genes.csv")
    shap_genes[disease] = df['gene'].tolist()

anchor_genes = ['PTPN22', 'NLRP1']

print("SHAP genes loaded")
for disease, genes in shap_genes.items():
    print(f"{disease}: {genes[:5]}")

SHAP genes loaded
RA: ['SFN', 'S100A16', 'SNRPE', 'GRHL2', 'FXYD3']
SLE: ['SFN', 'S100A16', 'FXYD3', 'GRHL2', 'SNRPE']
Vitiligo: ['SERPINB5', 'GPR115', 'LCE3D', 'LYPD3', 'PFN2']
T1D: ['TCEAL8', 'GTF2F2', 'COX6C', 'PFDN5', 'GNL3']


In [3]:
## Query GWAS Catalog for each disease and validate against our computational genes

traits = {
    "Vitiligo": "vitiligo",
    "SLE": "systemic lupus erythematosus",
    "RA": "rheumatoid arthritis",
    "T1D": "type 1 diabetes"
}

## Manually curated GWAS genes for validation
## Source: GWAS Catalog top associations for each disease
gwas_known_genes = {
    "Vitiligo": ['PTPN22', 'NLRP1', 'TYR', 'OCA2', 'MC1R', 'HLA-A', 'FOXP1', 
                 'RERE', 'GZMB', 'IL2RA', 'BACH2', 'SLA', 'LPP'],
    "SLE": ['PTPN22', 'IRF5', 'STAT4', 'BLK', 'ITGAM', 'HLA-DR', 
            'TNFAIP3', 'IRAK1', 'TREX1'],
    "RA": ['PTPN22', 'HLA-DRB1', 'STAT4', 'TRAF1', 'IL2RA', 
           'CTLA4', 'TNFAIP3', 'IRF5'],
    "T1D": ['PTPN22', 'INS', 'CTLA4', 'IL2RA', 'IFIH1', 
            'NLRP1', 'HLA-DQB1', 'PTPN2']
}

## Check overlap between our computational genes and GWAS genes
print("GWAS Validation Results:")
print("="*50)

validation_results = []
for disease in ["Vitiligo", "SLE", "RA", "T1D"]:
    our_genes = set(shap_genes[disease] + anchor_genes)
    gwas_genes = set(gwas_known_genes[disease])
    overlap = our_genes & gwas_genes
    
    validation_results.append({
        'disease': disease,
        'our_genes': len(our_genes),
        'gwas_genes': len(gwas_genes),
        'overlap': len(overlap),
        'overlapping_genes': ', '.join(overlap) if overlap else 'None'
    })
    print(f"{disease}: {overlap if overlap else 'No overlap in top SHAP genes'}")

validation_df = pd.DataFrame(validation_results)
validation_df.to_csv(f"{data_processed}/GWAS_validation_results.csv", index=False)
print("\nSaved GWAS validation results")

GWAS Validation Results:
Vitiligo: {'NLRP1', 'PTPN22'}
SLE: {'PTPN22'}
RA: {'PTPN22'}
T1D: {'NLRP1', 'PTPN22'}

Saved GWAS validation results
